In [ ]:
# Auto reload .py files when changed
%load_ext autoreload
%autoreload 2

##  Step 1 — Configuration

All project settings in one place.  
Every parameter used across LSTM, Environment, and RL agent comes from here.

| Parameter | Value | Reason |
|-----------|-------|--------|
| safe_temp_max | 27.0°C | ASHRAE A1 standard |
| critical_temp | 35.0°C | Class A2 maximum |
| w_overheat | 10.0 | Safety is highest priority |
| w_safe_bonus | 5.0 | Strong reward for safe zone |
| lstm_lookback | 8 | 2 hours of 15-min history |
| seed | 42 | Reproducibility |

In [ ]:
# Step 1 — Import and verify config
from configs.default_config import CoolSyncConfig

config = CoolSyncConfig()

print("✅ Config loaded successfully")
print(f"Safe temp range:  {config.safe_temp_min}°C → {config.safe_temp_max}°C")
print(f"Critical temp:    {config.critical_temp}°C")
print(f"Reward weights:   overheat={config.w_overheat} | energy={config.w_energy}")
print(f"LSTM lookback:    {config.lstm_lookback} steps = {config.lstm_lookback * 15} minutes")
print(f"Episode length:   {config.episode_length} steps")
print(f"Seed:             {config.seed}")

✅ Config loaded successfully
Safe temp range:  18.0°C → 27.0°C
Critical temp:    35.0°C
Reward weights:   overheat=10.0 | energy=0.2
LSTM lookback:    8 steps = 120 minutes
Episode length:   200 steps
Seed:             42


##  Step 2 — Data Check

Verifying both datasets are loaded correctly before building anything.

### Dataset 1: merged_lstm_core.csv
- Used to **train the LSTM** thermal predictor
- 27,013 rows at 15-minute intervals
- Jan 2024 → Oct 2024

### Dataset 2: stage4_cooling_control_norm.csv
- Used to **drive the RL environment**
- 3,498 rows at 1-hour intervals
- Jan 2025 → May 2025

>  All values are z-score normalized (mean=0, std=1)

In [ ]:
# Step 2 — Data Check
from configs.data_check import check_datasets

df1, df2 = check_datasets()

CHECKING: merged_lstm_core.csv
Rows:        27013
Columns:     36
Start:       2024-01-01 00:00:00
End:         2024-10-08 09:00:00
Null values: 0 (must be 0)

Key columns:
  requests_per_15min:
    min=10.000 | max=291.000 | mean=111.957
  total_tokens_15min:
    min=1440.000 | max=41906.000 | mean=16122.288
  avg_gpu_power_w:
    min=2.456 | max=71.460 | mean=27.492
  TLHC:
    min=-2.584 | max=1.787 | mean=-0.000

Time features:
  hour_sin: -1.000 to 1.000
  hour_cos: -1.000 to 1.000
  DoW:      [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
  WeH:      [np.int64(0), np.int64(1)]

CHECKING: stage4_cooling_control_norm.csv
Rows:        3498
Columns:     12
Start:       2025-01-01 00:00:00
End:         2025-05-26 17:00:00
Null values: 0 (must be 0)

Cooling actions:
Cooling_Strategy_Action
Eco Mode            718
Boost All           711
Increase Chiller    703
Reduce AHU          685
Maintain            681
Name: count, dtype: int64

Action

## Step 3 — LSTM Data Loader

The raw CSV cannot be fed directly into the LSTM.
This step handles three things:

1. Load merged_lstm_core.csv and select the 6 input features
2. Normalize features to 0-1 range so no single feature dominates
3. Build sliding window sequences of 8 steps (2 hours) per input

The LSTM learns: given the last 2 hours of token volume,
request rate, GPU power, and time features, predict the next
thermal load value (TLHC).

In [ ]:
# Step 3 — LSTM Data Loader

from forecasting.data_loader import LSTMDataLoader

loader = LSTMDataLoader(
    csv_path = 'data/merged_lstm_core.csv',
    lookback = 8
)

train_loader, val_loader = loader.get_loaders(
    batch_size = 32,
    val_split  = 0.2
)

print()
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print()
print("Data loader ready.")

Loaded      : 27013 rows
Period      : 2024-01-01 00:00:00 to 2024-10-08 09:00:00
Sequences   : 27005
Input shape : (27005, 8, 10)  (sequences, lookback, features)
Target shape: (27005,)
Train size  : 21604 sequences
Val size    : 5401 sequences

Train batches : 676
Val batches   : 169

Data loader ready.


## Step 4 — LSTM Model Architecture

The LSTM network learns to predict future thermal load
from 2 hours of historical data.

Architecture:
- Input: 8 timesteps x 6 features
- LSTM Layer 1: 64 hidden units
- LSTM Layer 2: 64 hidden units
- Dropout: 20% (prevents overfitting)
- Output Layer: 64 -> 1 (predicted TLHC)

Only the final timestep output is used for prediction
because it contains information from all previous steps.

In [ ]:
# Step 4 — LSTM Model

import torch
from forecasting.lstm_model import ThermalLSTM
from configs.default_config import CoolSyncConfig

config = CoolSyncConfig()

model = ThermalLSTM(
    input_size  = config.lstm_input_size,
    hidden_size = config.lstm_hidden,
    num_layers  = config.lstm_layers,
    dropout     = config.lstm_dropout,
)

print("Model architecture:")
print(model)
print()
print(f"Total trainable parameters: {model.count_parameters():,}")
print()

# Test with a fake input to verify shapes work correctly
fake_input = torch.randn(4, config.lstm_lookback,
                         config.lstm_input_size)
fake_output = model(fake_input)

print(f"Test input shape  : {fake_input.shape}")
print(f"Test output shape : {fake_output.shape}")
print()
print("Shape check passed. Model is ready.")

Model architecture:
ThermalLSTM(
  (lstm): LSTM(10, 64, num_layers=2, batch_first=True, dropout=0.2)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

Total trainable parameters: 52,801

Test input shape  : torch.Size([4, 8, 10])
Test output shape : torch.Size([4])

Shape check passed. Model is ready.


## Step 5 — Train LSTM

Training the ThermalLSTM on 27,013 rows of real GPU and token data.

The model learns to predict future thermal load (TLHC)
from the last 2 hours of token volume, request rate,
GPU power, and time features.

Key training decisions:
- Loss function  : MSE (mean squared error)
- Optimizer      : Adam with learning rate 0.001
- Early stopping : stops if no improvement for 10 epochs
- Best model     : saved only when validation loss improves
- Scheduler      : reduces learning rate when loss plateaus

In [ ]:
from forecasting.train_lstm import train_lstm

best_loss = train_lstm(
    csv_path      = 'data/merged_lstm_core.csv',
    epochs        = 100,
    learning_rate = 1e-4,
    batch_size    = 128,
    lookback      = 8,
    save_path     = 'results/checkpoints/lstm_best.pth'
)

print(f"\nTraining complete.")
print(f"Best validation loss : {best_loss:.4f}")

Device         : cpu

--- Loading Data ---
Loaded      : 27013 rows
Period      : 2024-01-01 00:00:00 to 2024-10-08 09:00:00
Sequences   : 27005
Input shape : (27005, 8, 10)  (sequences, lookback, features)
Target shape: (27005,)
Train size  : 21604 sequences
Val size    : 5401 sequences

--- Building Model ---
Parameters     : 52,801

--- Training ---
Epoch   1/100 | Train Loss: 0.5276 | Val Loss: 1.4916 | Best: 1.4916
